In [ ]:
from collections import Counter
import glob
import json
import os
import shutil
import xml.etree.ElementTree as ET
import xml.dom.minidom

from docx import Document
import numpy as np
import pandas as pd
from tqdm import tqdm
import win32com.client as win32

In [ ]:
ns_raw = 'xmlns:wpc="http://schemas.microsoft.com/office/word/2010/wordprocessingCanvas" xmlns:cx="http://schemas.microsoft.com/office/drawing/2014/chartex" xmlns:cx1="http://schemas.microsoft.com/office/drawing/2015/9/8/chartex" xmlns:cx2="http://schemas.microsoft.com/office/drawing/2015/10/21/chartex" xmlns:cx3="http://schemas.microsoft.com/office/drawing/2016/5/9/chartex" xmlns:cx4="http://schemas.microsoft.com/office/drawing/2016/5/10/chartex" xmlns:cx5="http://schemas.microsoft.com/office/drawing/2016/5/11/chartex" xmlns:cx6="http://schemas.microsoft.com/office/drawing/2016/5/12/chartex" xmlns:cx7="http://schemas.microsoft.com/office/drawing/2016/5/13/chartex" xmlns:cx8="http://schemas.microsoft.com/office/drawing/2016/5/14/chartex" xmlns:mc="http://schemas.openxmlformats.org/markup-compatibility/2006" xmlns:aink="http://schemas.microsoft.com/office/drawing/2016/ink" xmlns:am3d="http://schemas.microsoft.com/office/drawing/2017/model3d" xmlns:o="urn:schemas-microsoft-com:office:office" xmlns:oel="http://schemas.microsoft.com/office/2019/extlst" xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships" xmlns:m="http://schemas.openxmlformats.org/officeDocument/2006/math" xmlns:v="urn:schemas-microsoft-com:vml" xmlns:wp14="http://schemas.microsoft.com/office/word/2010/wordprocessingDrawing" xmlns:wp="http://schemas.openxmlformats.org/drawingml/2006/wordprocessingDrawing" xmlns:w10="urn:schemas-microsoft-com:office:word" xmlns:w="http://schemas.openxmlformats.org/wordprocessingml/2006/main" xmlns:w14="http://schemas.microsoft.com/office/word/2010/wordml" xmlns:w15="http://schemas.microsoft.com/office/word/2012/wordml" xmlns:w16cex="http://schemas.microsoft.com/office/word/2018/wordml/cex" xmlns:w16cid="http://schemas.microsoft.com/office/word/2016/wordml/cid" xmlns:w16="http://schemas.microsoft.com/office/word/2018/wordml" xmlns:w16du="http://schemas.microsoft.com/office/word/2023/wordml/word16du" xmlns:w16sdtdh="http://schemas.microsoft.com/office/word/2020/wordml/sdtdatahash" xmlns:w16sdtfl="http://schemas.microsoft.com/office/word/2024/wordml/sdtformatlock" xmlns:w16se="http://schemas.microsoft.com/office/word/2015/wordml/symex" xmlns:wpg="http://schemas.microsoft.com/office/word/2010/wordprocessingGroup" xmlns:wpi="http://schemas.microsoft.com/office/word/2010/wordprocessingInk" xmlns:wne="http://schemas.microsoft.com/office/word/2006/wordml" xmlns:wps="http://schemas.microsoft.com/office/word/2010/wordprocessingShape"'.split()
ms_ns = {ns.split("=")[0][6:]: ns.split("=")[1].strip('"') for ns in ns_raw}
[ET.register_namespace(prefix, uri) for prefix, uri in ms_ns.items()];

In [ ]:
scale = "One Inch"

### Convert doc files to docx

Easy to use docx package only handles docx files so convert .doc files to docx using win32com

In [ ]:
def convert_doc_to_docx(input_path: str|os.PathLike, output_path: str|os.PathLike) -> None:
    """Convert a .doc file at input_path to a .docx file at output_path

    Args:
        input_path (str | os.PathLike): Input .doc file to convert to docx
        output_path (str | os.PathLike): Output filepath for converted .docx file
    """
    word = win32.gencache.EnsureDispatch('Word.Application')
    doc = word.Documents.Open(input_path)    
    doc.SaveAs(output_path, FileFormat=16)  # 16 represents DOCX format
    doc.Close()
    word.Quit()

In [ ]:
doc_files = glob.glob(f"C:\\Users\\hlloyd\\projects\\union-lists\\data\\raw\\{scale}\\*.doc")[:-1]
if scale == "Half Inch":
    assert len(doc_files) == 38
    assert "Y102 53 X9052" in doc_files[-1]
elif scale == "One Inch":
    doc_files.remove('C:\\Users\\hlloyd\\projects\\union-lists\\data\\raw\\One Inch\\SIList48to72.doc')
    doc_files.remove('C:\\Users\\hlloyd\\projects\\union-lists\\data\\raw\\One Inch\\Y101RE~1.doc')
    assert len(doc_files) == 44
    assert "Y101 53 X9053" in doc_files[-1]
elif scale == "Quarter Inch":
    y_files = [d for d in doc_files if "Y104 " in d]
    doc_files = [d for d in doc_files if "C." in d] + y_files
    assert len(doc_files) == 39
    assert "Y104 53 X9051" in doc_files[-1]

In [ ]:
with tqdm(doc_files) as t:
    for f in doc_files:
        doc_id = os.path.basename(f[:-4])
        t.set_description(doc_id)
        convert_doc_to_docx(f, f.replace("raw", "interim") + "x")
        t.update()

### Remove smart tags from word docs

In [ ]:
def find_and_replace(doc_path: str|os.PathLike, find_text: str, replace_text: str):
    """Use win32com to run find and replace on a Word document at doc_path
    Used to find/replace text in union list documents, removing smart tags in the process

    Function compiled by Claude
    find.Execute args modified by HL
    Documentation written by HL
    Args:
        doc_path (str | os.PathLike): Path to a .docx file to carry out find/replace on 
        find_text (str): Text to find
        replace_text (str): Text to replace
    """
    word = win32.Dispatch("Word.Application")
    word.Visible = False

    doc = word.Documents.Open(doc_path)

    find = doc.Content.Find
    find.ClearFormatting()
    find.Replacement.ClearFormatting()

    find.Execute(
        FindText=find_text,
        ReplaceWith=replace_text,
        Replace=2,
        MatchCase=True,
        MatchWholeWord=True,
        MatchWildcards=False,
        Forward=True,
        Wrap=1
    )

    doc.Save()
    doc.Close()
    word.Quit()

In [ ]:
def count_smart_tags(root: xml.etree.ElementTree.Element, ns: dict[str, str]) -> collections.Counter:
    """Identify and count instances of text in smartTag tags in an XML document

    Args:
        root (xml.etree.ElementTree.Element): The root Element of the xml to search
        ns (dict[str, str]): A namespace dictionary for resolving the namespaces in an xml

    Returns:
        collections.Counter: A Counter dict of counts of instances of text in smartTags in root
    """
    smart_tag_text = [x.text for x in root.findall(f".//{{{ns['w']}}}smartTag/{{{ns['w']}}}r/{{{ns['w']}}}t")]
    return Counter(smart_tag_text)

In [ ]:
docx_files = glob.glob(f"C:\\Users\\hlloyd\\projects\\union-lists\\data\\interim/{scale}/*.docx")
docx_files = [x for x in docx_files if "~" not in x and "(2)" not in x]
docx_files = [x for x in docx_files if "_mod" not in x]

if scale == "Half Inch":
    assert len(doc_files) == 38
elif scale == "One Inch":
    assert len(doc_files) == 44
elif scale == "Quarter Inch":
    assert len(doc_files) == 39

- unpack all docx files to access underlying xml
- count_smart_tags() with xml, win32com failed to find in .docx, possibly because embedded in tables
- find_and_replace() all smartTag text with the same text, removing smartTags in the process
- re-unpack all .docx and check smartTag count is 0, if so delete intermediate unpacked folders

In [ ]:
for f in docx_files:
    shutil.copy2(f, f[:-5] + "_c.zip")
    shutil.unpack_archive(f[:-5] + "_c.zip", f[:-5] + '_c')

In [ ]:
# Have to calculate smart tag ids and counts from the xml, win32com failed to find, possibly because embedded in tables
smart_tag_counts = {}
for f in docx_files:
    doc_id = os.path.basename(f)[:-5]
    with open(os.path.join(f[:-5] + "_c", "word\\document.xml"), encoding="utf8") as f:
        tree = ET.parse(f)
        root = tree.getroot()

    smart_tags = count_smart_tags(root, ms_ns)
    smart_tag_counts[doc_id] = smart_tags

# with open("../data/interim/smart_tag_counts.json", "w", encoding="utf8") as f:
#     json.dump(smart_tag_counts, f, indent=4)

In [ ]:
smart_tag_counts

In [ ]:
with tqdm(docx_files) as t:
    for f in docx_files:
        doc_id = os.path.basename(f[:-5])
        t.set_description(doc_id)
        smart_tags = smart_tag_counts[doc_id]
        if not smart_tags:
            t.update()
            continue
    
        for tag in smart_tags:
            find_and_replace(f, tag, tag)
        t.update()

In [ ]:
for f in docx_files:
    shutil.copy2(f, f[:-5] + "_mod.zip")
    shutil.unpack_archive(f[:-5] + "_mod.zip", f[:-5] + '_mod')

In [ ]:
# check removed
mod_smart_tag_counts = {}
for f in docx_files:
    doc_id = os.path.basename(f)[:-5]
    with open(os.path.join(f[:-5] + "_mod", "word\\document.xml"), encoding="utf8") as f:
        tree = ET.parse(f)
        root = tree.getroot()

    mod_smart_tags = count_smart_tags(root, ms_ns)
    mod_smart_tag_counts[doc_id] = mod_smart_tags

assert sum([len(v) for v in mod_smart_tag_counts.values()]) == 0

In [ ]:
if sum([len(v) for v in mod_smart_tag_counts.values()]) == 0:
    for f in docx_files:
        raw_xml_folder = f[:-5] + '_c'
        raw_zip = f[:-5] + '_c.zip'
        mod_xml_folder = f[:-5] + '_mod'
        mod_zip = f[:-5] + '_mod.zip'
        
        if os.path.exists(raw_xml_folder):
            shutil.rmtree(raw_xml_folder)

        if os.path.exists(raw_zip):
            os.remove(raw_zip)

        if os.path.exists(mod_xml_folder):
            shutil.rmtree(mod_xml_folder)

        if os.path.exists(mod_zip):
            os.remove(mod_zip)

### Extract tables from docx files

In [ ]:
docx_files = glob.glob(f"C:\\Users\\hlloyd\\projects\\union-lists\\data\\interim/{scale}/*.docx")
docx_files = [x for x in docx_files if "~" not in x and "(2)" not in x]
docx_files = [x for x in docx_files if "_mod" not in x]
if scale == "Half Inch":
    assert len(doc_files) == 38
elif scale == "One Inch":
    assert len(doc_files) == 44
elif scale == "Quarter Inch":
    assert len(doc_files) == 39

In [ ]:
all_data = {}
for f in docx_files:
    table = Document(f).tables[0]
    data = {i: table.column_cells(i) for i, _ in enumerate(table.columns)}
    all_data[os.path.basename(f).split(".")[0]] = data

In [ ]:
dfs = {k: pd.DataFrame(v).apply(lambda x: x.apply(lambda y: y.text)) for k,v in all_data.items()}

In [ ]:
def extract_headers(row):
    headers = {
        "Post-1905": [[], None],
        "1886-1905": [[], None],
        "Pre-1886": [[], None]
    }

    for i, cell in zip(row.index, row):
        if "Post-1905" in cell:
            headers["Post-1905"][0].append(i)
            headers["Post-1905"][1] = cell
        elif "1886-" in cell:
            headers["1886-1905"][0].append(i)
            headers["1886-1905"][1] = cell
        elif "Pre-1886" in cell:
            headers["Pre-1886"][0].append(i)
            headers["Pre-1886"][1] = cell
        else:
            raise ValueError(f"No date information in cell: {cell}")

    return headers
        
def clean_map_df(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, str]]:
    
    if len(df) == 6 and df.apply(lambda x: x.str.contains("are known to have")).any().any():
        no_known_text = np.unique(df.apply(lambda x: x[x.str.contains("are known to have")]).values)[0]
        df = pd.DataFrame()
        table_metadata = {"no_known_maps": no_known_text}
        return df, table_metadata

    dup_cols = []
    for col in df.columns[:-1]:
        if np.array_equal(df[col].values, df[col+1].values):
            dup_cols.append(col + 1)

    df = df.drop(columns=dup_cols)
    
    if df.shape[1] > 2:       
        long_lines_idx = df.apply(lambda x: x.transform(len)).sum(axis=1).sort_values(ascending=False).iloc[:2].index
        header_row, footer_row = long_lines_idx.min(), long_lines_idx.max()

        headers = extract_headers(df.loc[header_row])
        post_1905_header, mid_header, pre_1886_header = headers["Post-1905"][1], headers["1886-1905"][1], headers["Pre-1886"][1]
        post_1905_footer, mid_footer, pre_1886_footer = df.loc[footer_row,[headers["Post-1905"][0][0], headers["1886-1905"][0][0], headers["Pre-1886"][0][0]]]

        df = df.where(lambda x: x != '', pd.NA).dropna(how="all", axis=0)
        old_idx = df.index.to_list()
        iloc_header_row, iloc_footer_row = old_idx.index(header_row), old_idx.index(footer_row)
        df = df.reset_index(drop=True)
        
        if header_row > 9 or len(df) - footer_row > 10:
            raise ValueError(f"Header or footer in unlikely location header: {header}, footer: {footer}")
        
        table_metadata = {"Post-1905_header": post_1905_header, "1886-1905_header": mid_header, "Pre-1886_header": pre_1886_header,
                          "Post-1905_footer": post_1905_footer, "1886-1905_footer": mid_footer, "Pre-1886_footer": pre_1886_footer}
        
        df = df.iloc[iloc_header_row + 1: iloc_footer_row].copy()
        cols = []
        for k, v in headers.items():
            [cols.append(k + f"_{i + 1}") for i, col in enumerate(v[0])]
        df.columns = cols
        df = df.reindex(columns=['Post-1905_1', 'Post-1905_2', '1886-1905_1', '1886-1905_2', 'Pre-1886_1', 'Pre-1886_2'])
        
    elif df.shape[1] == 2:
        df = df.dropna(how="all").reset_index(drop=True)
        df.columns = ["Post-1905_1", "metadata"]
        table_metadata = {}
    else:
        raise ValueError(f"df has unexpected shape {df.shape}")

    df = df.apply(lambda x: x.str.strip().str.rstrip("\n"), axis=1)
    return df, table_metadata

In [ ]:
for file_id, df in dfs.items():
    df, metadata = clean_map_df(df)
    if not df.empty:
        df.to_csv(f"../data/interim/{scale}/{file_id}.csv", encoding="utf-8-sig", index=False)
        with open(f"../data/interim/{scale}/{file_id}.json", "w",encoding="utf8") as f:
            json.dump(metadata, f)

In [ ]:
df, metadata = clean_map_df(dfs["38B"])
df

In [ ]:
# df.to_csv(f"../data/interim/{scale}/38B.csv", encoding="utf8", index=False)
# with open(f"../data/interim/{scale}/38B.json", "w",encoding="utf8") as f:
#     json.dump(metadata, f)

### Pretty print the docx xml to diagnose issues

In [ ]:
dom = xml.dom.minidom.parse(f"C:\\Users\\hlloyd\\projects\\union-lists\\data\\interim\\{scale}\\34B - Copy\\word\\document.xml")

In [ ]:
with open("../data/interim/34B_indent.xml", "w", encoding="utf8") as f:
    f.write(dom.toprettyxml())

### Direct XML modification

hard to make work in practice, difficult to recompile xml to working word doc

In [ ]:
def remove_smart_tags(root, ns):   
    smart_tags_6deep_parents = root.findall(f"./*/*/*/*/*/{{{ns['w']}}}smartTag/..")
    print(f"{len(smart_tags_6deep_parents)} smart tags at depth 6 found")
    for i, par in enumerate(smart_tags_6deep_parents):
        st = par.findall(f"{{{ns['w']}}}smartTag")[0]
        if st not in par:
            raise ValueError("smartTag not in parent's subelements")
        st_elems = [x for x in st]
        if len(st_elems) > 1:
            raise ValueError(f"Too many elements in smartTag, found {len(st_elems)}")
        par.append(st_elems[0])
        par.remove(st)

    if len(root.findall(f"./*/*/*/*/*/{{{ns['w']}}}smartTag")) > 0:
        print(f"{root} recurring")
        remove_smart_tags(root, ms_ns)
    else:
        return None

In [ ]:
for f in tqdm(docx_files):
    print(f)
    with open(os.path.join(f[:-5] + "_c", "word\\document.xml"), "r", encoding="utf-8") as g:
        tree = ET.parse(g)
        root = tree.getroot()
        remove_smart_tags(root, ms_ns)
    with open(os.path.join(f[:-5] + "_c", "word\\document.xml"), "wb") as g:
        tree.write(g, encoding="utf-8", xml_declaration=True)
    print("\n")

In [ ]:
for f in docx_files:
    shutil.make_archive(base_name=f[:-5] + "_mod", format="zip", root_dir=f[:-5] + '_c')
    shutil.copy2(f[:-5] + "_mod.zip", f[:-5] + "_mod.docx")